# Train with Full Data Lineage (DVC ➜ Feast ➜ MLflow)

This notebook is the third step in the NYC taxi demo. It assumes:

1. `ingest_taxi_dvc.py` has produced a versioned cleaned parquet under `~/shared/demo/test-data/nyc-taxi/`.
2. `materialize_features.py` has registered + materialized a Feast feature view `zone_hourly_stats` under `~/shared/demo/test-data/feast-repo/`.

What we do here:

- Sample raw trips to build a labeled training set (predict `tip_amount > $3`).
- Pull historical, point-in-time features from Feast using the lagged pickup time (so we look at the *previous* hour's zone stats — no leakage).
- Train a sklearn classifier with MLflow autologging.
- Tag the run with `dvc_commit`, `feast_project`, `feast_feature_view`, and `feast_registry_mtime` so the model can be traced back to the exact data + features it was trained on, months later.
- Reload the model from MLflow and confirm the lineage tags are intact.

In [ ]:
# !pip install -q feast mlflow scikit-learn   # uncomment if not already installed

In [ ]:
import os
import subprocess
from datetime import datetime, timezone
from pathlib import Path

import mlflow
import pandas as pd
from feast import FeatureStore
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

BASE = Path(os.environ.get("SHARED_STORAGE", "/home/jovyan/shared/demo/test-data"))
DVC_REPO = BASE / "nyc-taxi"
RAW_PARQUET = DVC_REPO / "cleaned/taxi.parquet"
FEAST_REPO = BASE / "feast-repo"
MLRUNS = BASE / "mlruns"

MLRUNS.mkdir(parents=True, exist_ok=True)
mlflow.set_tracking_uri(f"file:{MLRUNS}")
mlflow.set_experiment("nyc_taxi_tipping")

## 1. Capture upstream lineage

Pull the current DVC commit hash and the Feast registry mtime so we can pin this training run to exact upstream versions.

In [ ]:
dvc_commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=DVC_REPO, text=True
).strip()

registry_path = FEAST_REPO / "data/registry.db"
registry_mtime = datetime.fromtimestamp(registry_path.stat().st_mtime, tz=timezone.utc).isoformat()

store = FeatureStore(repo_path=str(FEAST_REPO))
fv = store.get_feature_view("zone_hourly_stats")

lineage_tags = {
    "dvc_commit": dvc_commit,
    "feast_project": store.project,
    "feast_feature_view": fv.name,
    "feast_registry_mtime": registry_mtime,
    "source_parquet": str(RAW_PARQUET),
}
lineage_tags

## 2. Build labeled training set

Sample 50k trips. Use lagged pickup time (`-1 hour`) as the event timestamp so Feast returns the *previous* hour's zone stats — that's what would have been available at prediction time.

In [ ]:
trips = pd.read_parquet(
    RAW_PARQUET,
    columns=["tpep_pickup_datetime", "PULocationID", "fare_amount", "trip_distance", "tip_amount"],
)
trips = trips[(trips["tpep_pickup_datetime"] >= "2024-01-01") & (trips["tpep_pickup_datetime"] < "2024-02-01")]
trips = trips[(trips["fare_amount"] > 0) & (trips["trip_distance"] > 0)]
trips = trips.sample(50_000, random_state=0).reset_index(drop=True)

trips["high_tip"] = (trips["tip_amount"] > 3).astype(int)
trips["event_timestamp"] = (trips["tpep_pickup_datetime"] - pd.Timedelta(hours=1)).dt.tz_localize("UTC")

entity_df = trips[["PULocationID", "event_timestamp", "high_tip", "trip_distance", "fare_amount"]]
print(f"{len(entity_df):,} trips, positive rate = {entity_df['high_tip'].mean():.1%}")
entity_df.head()

## 3. Pull point-in-time features from Feast

Feast does the time-travel join: for each `(PULocationID, event_timestamp)` it finds the most recent `zone_hourly_stats` row at or before that timestamp.

In [ ]:
training_df = store.get_historical_features(
    entity_df=entity_df,
    features=[
        "zone_hourly_stats:avg_fare",
        "zone_hourly_stats:avg_distance",
        "zone_hourly_stats:avg_passengers",
        "zone_hourly_stats:trip_count",
    ],
).to_df()

training_df = training_df.dropna()
feature_cols = ["avg_fare", "avg_distance", "avg_passengers", "trip_count", "trip_distance", "fare_amount"]
X = training_df[feature_cols]
y = training_df["high_tip"]
print(f"{len(training_df):,} rows after Feast join")
training_df.head()

## 4. Train + log to MLflow with lineage tags

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0, stratify=y)

mlflow.sklearn.autolog(log_models=True, log_input_examples=False)

with mlflow.start_run(run_name="rf_baseline") as run:
    for k, v in lineage_tags.items():
        mlflow.set_tag(k, v)
    mlflow.log_dict(lineage_tags, "lineage.json")

    model = RandomForestClassifier(n_estimators=50, max_depth=8, n_jobs=-1, random_state=0)
    model.fit(X_train, y_train)

    auc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
    mlflow.log_metric("test_auc", auc)

    run_id = run.info.run_id
    print(f"run_id = {run_id}")
    print(f"test AUC = {auc:.3f}")

## 5. Reproduce months later

Given just the run_id, recover the model and trace back to the exact dataset and feature view it was trained on.

In [ ]:
loaded = mlflow.sklearn.load_model(f"runs:/{run_id}/model")
client = mlflow.tracking.MlflowClient()
tags = {k: v for k, v in client.get_run(run_id).data.tags.items() if not k.startswith("mlflow.")}

print("Recovered lineage:")
for k, v in tags.items():
    print(f"  {k}: {v}")

print(f"\nTo restore the exact training dataset:")
print(f"  cd {DVC_REPO} && git checkout {tags['dvc_commit']} && dvc pull")
print(f"\nLoaded model: {type(loaded).__name__} with {loaded.n_estimators} trees")

## 6. Browse runs in the MLflow UI

From a terminal on the same host:

```bash
mlflow ui --backend-store-uri file:$HOME/shared/demo/test-data/mlruns --host 0.0.0.0 --port 5000
```

Then open the URL Saturn proxies for port 5000 and click into the run — the **Tags** panel shows the four lineage tags we set, the **Artifacts** tab has the model and `lineage.json`, and the **Parameters/Metrics** are populated by `autolog`.